## Setup and Import Libraries

In [1]:
import os
from langchain_community.graphs import Neo4jGraph
from dotenv import load_dotenv

import warnings
warnings.filterwarnings("ignore")

In [2]:
load_dotenv()

True

In [3]:
NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE')

In [4]:
knowledge_graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE
)

## Creating Movie Knowledge Graph

In [5]:
movie_query ="""
LOAD CSV WITH HEADERS FROM
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' as row

MERGE(m:Movie{id:row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)
FOREACH (director in split(row.director, '|') | 
    MERGE (p:Person {name:trim(director)})
    MERGE (p)-[:DIRECTED]->(m))
FOREACH (actor in split(row.actors, '|') | 
    MERGE (p:Person {name:trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m))
FOREACH (genre in split(row.genres, '|') | 
    MERGE (g:Genre {name:trim(genre)})
    MERGE (m)-[:IN_GENRE]->(g))
"""

In [9]:
result = knowledge_graph.query(movie_query)

## Query Movie Knowledge Graph

In [10]:
cypher = """
    MATCH (n) 
    RETURN count(n)
"""

result = knowledge_graph.query(cypher)
result

[{'count(n)': 1557}]

In [11]:
cypher = """
    MATCH (n) 
    RETURN count(n) AS numberOfNodes
"""

result = knowledge_graph.query(cypher)
result

[{'numberOfNodes': 1557}]

In [12]:
print(f"There are {result[0]['numberOfNodes']} nodes in this graph.")

There are 1557 nodes in this graph.


### Match Movie Nodes

In [13]:
cypher = """
    MATCH (m:Movie) 
    RETURN count(m) AS numberOfMovies
"""

result = knowledge_graph.query(cypher)
result

[{'numberOfMovies': 299}]

### Match Person Nodes

In [14]:
cypher = """
    MATCH (p:Person) 
    RETURN count(p) AS numberOfPeople
"""

result = knowledge_graph.query(cypher)
result

[{'numberOfPeople': 1239}]

### Match by Person

In [15]:
cypher = """
    MATCH (tom:Person {name:"Tom Hanks"}) 
    RETURN tom
"""

result = knowledge_graph.query(cypher)
result

[{'tom': {'name': 'Tom Hanks'}}]

### Match by Movie

In [16]:
cypher = """
    MATCH (movie:Movie {title:"Toy Story"}) 
    RETURN movie
"""

result = knowledge_graph.query(cypher)
result

[{'movie': {'imdbRating': 8.3,
   'id': '1',
   'title': 'Toy Story',
   'released': neo4j.time.Date(1995, 11, 22)}}]

In [17]:
cypher = """
    MATCH (movie:Movie {title:"Toy Story"}) 
    RETURN movie.released
"""

result = knowledge_graph.query(cypher)
result

[{'movie.released': neo4j.time.Date(1995, 11, 22)}]

In [18]:
cypher = """
    MATCH (movie:Movie {title:"Toy Story"}) 
    RETURN movie.released, movie.imdbRating
"""

result = knowledge_graph.query(cypher)
result

[{'movie.released': neo4j.time.Date(1995, 11, 22), 'movie.imdbRating': 8.3}]

### Conditional Matching

In [20]:
cypher = """
    MATCH (nineties:Movie) 
    WHERE nineties.released >= date("1990-01-01") 
    AND nineties.released < date("2000-01-01") 
    RETURN nineties.title
"""

result = knowledge_graph.query(cypher)
result

[{'nineties.title': 'Toy Story'},
 {'nineties.title': 'Jumanji'},
 {'nineties.title': 'Grumpier Old Men'},
 {'nineties.title': 'Waiting to Exhale'},
 {'nineties.title': 'Father of the Bride Part II'},
 {'nineties.title': 'Heat'},
 {'nineties.title': 'Sabrina'},
 {'nineties.title': 'Tom and Huck'},
 {'nineties.title': 'Sudden Death'},
 {'nineties.title': 'GoldenEye'},
 {'nineties.title': 'American President, The'},
 {'nineties.title': 'Dracula: Dead and Loving It'},
 {'nineties.title': 'Balto'},
 {'nineties.title': 'Nixon'},
 {'nineties.title': 'Cutthroat Island'},
 {'nineties.title': 'Casino'},
 {'nineties.title': 'Sense and Sensibility'},
 {'nineties.title': 'Four Rooms'},
 {'nineties.title': 'Ace Ventura: When Nature Calls'},
 {'nineties.title': 'Money Train'},
 {'nineties.title': 'Get Shorty'},
 {'nineties.title': 'Copycat'},
 {'nineties.title': 'Assassins'},
 {'nineties.title': 'Powder'},
 {'nineties.title': 'Leaving Las Vegas'},
 {'nineties.title': 'Othello'},
 {'nineties.title': 

### Patterns with Multiple Nodes

In [21]:
cypher = """
    MATCH (person:Person)-[:ACTED_IN]->(movie:Movie) 
    RETURN person.name, movie.title LIMIT 10
"""

result = knowledge_graph.query(cypher)
result

[{'person.name': 'Jim Varney', 'movie.title': 'Toy Story'},
 {'person.name': 'Tim Allen', 'movie.title': 'Toy Story'},
 {'person.name': 'Tom Hanks', 'movie.title': 'Toy Story'},
 {'person.name': 'Don Rickles', 'movie.title': 'Toy Story'},
 {'person.name': 'Robin Williams', 'movie.title': 'Jumanji'},
 {'person.name': 'Bradley Pierce', 'movie.title': 'Jumanji'},
 {'person.name': 'Kirsten Dunst', 'movie.title': 'Jumanji'},
 {'person.name': 'Jonathan Hyde', 'movie.title': 'Jumanji'},
 {'person.name': 'Walter Matthau', 'movie.title': 'Grumpier Old Men'},
 {'person.name': 'Ann-Margret', 'movie.title': 'Grumpier Old Men'}]

In [22]:
cypher = """
    MATCH (person:Person {name: "Tom Hanks"})-[:ACTED_IN]->(movie:Movie) 
    RETURN person.name,movie. title
"""

result = knowledge_graph.query(cypher)
result

[{'person.name': 'Tom Hanks', 'movie.title': 'Toy Story'},
 {'person.name': 'Tom Hanks', 'movie.title': 'Apollo 13'}]

In [23]:
cypher = """
    MATCH (person:Person {name:"Tom Hanks"})-[:ACTED_IN]->(movie)<-[:ACTED_IN]-(coActors) 
    RETURN coActors.name, movie.title
"""

result = knowledge_graph.query(cypher)
result

[{'coActors.name': 'Jim Varney', 'movie.title': 'Toy Story'},
 {'coActors.name': 'Tim Allen', 'movie.title': 'Toy Story'},
 {'coActors.name': 'Don Rickles', 'movie.title': 'Toy Story'},
 {'coActors.name': 'Kevin Bacon', 'movie.title': 'Apollo 13'},
 {'coActors.name': 'Bill Paxton', 'movie.title': 'Apollo 13'},
 {'coActors.name': 'Gary Sinise', 'movie.title': 'Apollo 13'}]

## Delete data from Graph

In [24]:
cypher = """
    MATCH (person:Person {name:"Don Rickles"})-[actedIn:ACTED_IN]->(movie:Movie)
    RETURN person.name, movie.title
"""

result = knowledge_graph.query(cypher)
result

[{'person.name': 'Don Rickles', 'movie.title': 'Toy Story'}]

In [25]:
cypher = """
    MATCH (person:Person {name:"Don Rickles"})-[actedIn:ACTED_IN]->(movie:Movie)
    DELETE actedIn
"""

result = knowledge_graph.query(cypher)
result

[]

In [26]:
cypher = """
    MATCH (person:Person {name:"Don Rickles"})-[actedIn:ACTED_IN]->(movie:Movie)
    RETURN person.name, movie.title
"""

result = knowledge_graph.query(cypher)
result

[]

## Adding Data to Graph

In [27]:
cypher = """
    CREATE (person:Person {name:"Wallace Shawn"})
    RETURN person
"""

result = knowledge_graph.query(cypher)
result

[{'person': {'name': 'Wallace Shawn'}}]

In [28]:
cypher = """
    MATCH (person:Person {name:"Wallace Shawn"}), (another_person:Person {name:"Tim Allen"})
    MERGE (person)-[hasRelationship:WORKS_WITH]->(another_person)
    RETURN person, hasRelationship, another_person
"""

result = knowledge_graph.query(cypher)
result

[{'person': {'name': 'Wallace Shawn'},
  'hasRelationship': ({'name': 'Wallace Shawn'},
   'WORKS_WITH',
   {'name': 'Tim Allen'}),
  'another_person': {'name': 'Tim Allen'}}]